
---
REPOSITORY STRUCTURE -- The Big Picture
   The repo splits into two halves: STUFF THE ORGANISERS PROVIDE (evaluation
   infrastructre) and STUFF YOU IMPLEMENT (your policy).


  aic/
  ├── docs/                    # 12+ documentation guides
  │
  ├── ORGANIZER SIDE (evaluation):
  │   ├── aic_engine/          # Trail orchestration & validation       [DOCUMENTED]
  │   ├── aic_bringup/         # Launch files for simulation            [DOCUMENTED]
  │   ├── aic_controller/      # Low-level robot control (ros2_control)
  │   ├── aic_adapter/         # Sensor fusion & synchronisation
  │   ├── aic_gazebo/          # Gazebo physics plugins
  │   ├── aic_scoring/         # Automated scoring system
  │   ├── aic_description/     # URDF/SDF & environment models
  │   └── aic_assets/          # 3D models (meshes, textures)
  │
  ├── YOUR SIDE (participant):
  │   ├── aic_model/           # Policy framework template (implement this!)
  │   ├── aic_example_policies/# Reference implementations to learn from [DOCUMENTATION]
  │   └── aic_interfaces/      # ROS 2 message/service/action definitions
  │
  ├── aic_utils/               # Utilities (teleoperation, MuJuCo, LeRobot)
  ├── docker/                  # Container definitions for submission
  ├── pixi.toml                # Dependency manager config
  └── aic.repos                # External dependency manifest


---
THE CORE PACKAGES IN DETAIL
   
1. `aic_model/` - YOUR POLICY FRAMEWORK
   This is the template you build on. It has two key files:

   `aic_model.py` (275 lines) - A ROS 2 lifecycle node that handles all the
   boilerplate: loading your policy dynamically, managing lifecycle states,
   implementing the `/insert_cable` action server, and publishing motion
   commands.

   `policy.py` (146 lines) - The abstract base class you subclass. Your job is
   to implement one method:

```Python
class MyPolicy(Policy):
    def insert_cable(self, task, get_observation, move_robot, send_feedback):
        observation = get_observation()      # Sensor data @ 20 Hz
        # Process observation, compute target pose
        move_robot(motion_update=...)        # Send command to controller
        send_feedback("inserting cable...")
        return True                          # Task success
```
   The helpers you get: `set_pose_target()`, `time_now()`, `sleep_for()`, 
   `get_logger()`.



2. `aic_interfaces/` - COMMUNICATION PROTOCOLS
   These define hwo everything talks to each other via ROS 2:

   TASK INTERFACE:
   * `InsertCable.action` - The main action: Goal (Task) -> Result (success +
     message) + Feedback (string)
   * `Task.msg` - Contains: `cable_name`, `plug_name`, `plug_type`, `port_name`,
     `port_type`, `target_module_name`, `time_limit`

   CONTROL INTERFACE: 
   - `MotionUpdate.msg` - Cartesian impedance control (pose, velocity, stiffness
     matrix, damping matrix, feedforward wrench)
   - `JointMotionUpdate.msg` - Joint-space control (positions, velocities,
     accelerations + impedance params)
   - `ChangeTableMode.srv` - Switch between Cartesian and Joint control

   OBSERVATION INTERFACE:
   - `Observation.msg` - Composite at 20 Hz: 3 camera images, camera_info, joint
     states, wrist wrench (F/T), controller state (TCP pose/velocity, tracking
     error)



3. `aic_engine/` - TRAIL ORCHESTRATOR
   Automates the evaluation pipeline. For each trail it:
   1. Validates your model is a proper ROS 2 Lifecycle node
   2. Spawns the task board from YAML config
   3. Sends cable insertion tasks via `/insert_cable`
   4. Monitors execution
   5. Collects scoring data

   Configuration is YAML-driven:
```yaml
trials:
  trial_1:
    scene:
      task_board: {pose, nic_rails[], sc_rails[], mount_rails[]}
      cable: {cable_type, pose}
```
   
4. `aic_controller/` - ROBOT CONTROL
   Based on `ros2_control`. Two modes:
   - CARTESIAN IMPEDANCE CONTROL - Track TCP pose targets with tunable 
     stiffness/damping
   - JOINT-SPACE CONTROL - Direct joint position/velocity targets

   Features admittance control for interaction forces, F/T sensor integration,
   and real-time trajectory generation. 


4. `aic_adapter/` - SENSOR FUSION
   C++ node that subscribes to all async sensor streams (3 cameras, joint states
   , F/T sensor, controller state), time-aligns them in deque buffers, and
   publishes a composite `Obervsation` message at 20 Hz.


5. `aic_gazebo/` - PHYSICS PLUGINS
   Five Gazebo plugins:
   - ScoringPlugin - Publishes scoring data via Protobuf


   ROS-2 is a decentralised, modular middleware framework for robotics, 
   utilising DDS (Data Distribution Service) for robust, secure and real-time
   communication between ndoes. Key concepts include nodes (functional units),
   topics (publish-subscribe messaging), services (request-reply), actions
   (long-running tasks), and parameters (configuration), all facilitating
   scalable robot development.

CORE ROS-2 CONCEPTS EXPLAINED: 
   - NODES: Independent, specialised programs that perform specific tasks (e.g.,
     one node for camera data, another for motor control)
   - TOPICS: The primary method for data exchange. Nodes publish data to a topic
     (e.g., `/scan`) and subscribe to topics to receive data, allowing 
     many-to-many communication.
   - MESSAGES: The specific data types (structure) sent over topics (e.g., 
     `sensor_msgs/msg/LaserScan`).
   - SERVICES: A synchronous request/reply mechanism used when a node needs a 
     quick, immediate action from another (e.g., turning on a service).
   - ACTIONS: Asynchronous, pre-emptable, and long-running goal-based 
     communication. Ideal for complex tasks like navigating to a location, 
     providing feedback during execution. 
   - PARAMETERS: Configurable parameters (e.g., maximum speed) that can be
     changed dynamicaly at runtime.
   - ROS GRAPH: The network of all nodes and their connections to each other.
   - QUALITY OF SERVICE (QoS): Policies (reliability, durability) that allow
     developers to tune communication settings for specific needs, such as
     ensuring critical data is never lsot.
   - DISCOVERY: The automatic process by which nodes find and connect to
     other nodes in the network without a central master.     



   ... to understand your SO-101 robotic arm, you have to think of it as a 
   KINEMATIC CHAIN: a series of rigid bodies (links) connected by joints that
   provide constrained motion.

   In the real world, these aren't just abstract math; they are the literal
   "bones" and "muscles" of your robot.


---
1. SERIAL KINEMATIC CHAIN (Open Chain)   
   This is what your SO-101 arm is. It starts at a fixed base and ends at a 
   "free" tip (the gripper).
   * THE STRUCTURE: Each link is connected to the next in a single line, like a
     human arm (Shoulder --> Elbow --> Wrist).
   * THE DOWNSIDE: If the shoulder moves 1 degree, that error is magnified all
     the way down to the gripper. This why you noticed the 5090 is needed for
     high-fidelity physics; calculaing these compounding errors in Isaac Sim is
     computationally expensive. 


2. CLOSED KINEMATIC CHAIN
   A closed chain is formed when the loop is completed, meaning there is no
   "free end".
   * THE STRUCTURE: Every link is connected to at least two others, forming a
     loop. Think of a PARALLEL ROBOT (like a Delta robot over a conveyor belt) 
     or a BICYCLE (Pedal --> Chain --> Wheel --> Frame --> Pedal).
   * REAL-WORLD MODELLING:
      * HIGH-SPEED PICKING: Delta robots use closed chains because they are 
        incredibly stiff and fast.
      * LEGGED ROBOTICS: A quadruped (like Unitree or Spot) becomes a closed   
        kinematic chain the moment its foot touches the ground.
      * PROSTHETICS: Complex knee joints often use a "four-bar linkage" (a
        classic closed chain) to mimic human motion.


3. WHY DOES THIS MATTER FOR YOUR "LEGO ARM"?
   When your SO-101 (Open Chain) picks up a Lego Brick and SNAPS IT ONTO A 
   BASEPLATE, the physics state instantly changes:
      1. BEFORE SNAPPING: You are an OPEN CHAIN. The arm can move freely in
         space.
      2. DURING SNAPPING: The moment the brick clicks onto the fixed baseplate
         while the robot is still holding it, you have created a CLOSED 
         KINEMATIC CHAIN.
      3. THE CHALLENGE: Closed chains are much harder for Isaac Sim to solve
         because the movements of the joints are now mathematically "locked" to
         each other. If the "Elbow" moves, the "Shoulder" must move a specific 
         way because the "Hand" is stuck to the table.

    In `run_articulation.py`, the "Cart-Pole" is an open chain. But if the pole
    were attached to the ground at both ends, it would be a closed chain.      

---


---
-- A ROS2 lifecycle node (or managed node) is a specialised node that follows a
   strict state machine--unconfigured, inactive, active, and finalised--to 
   manage its own lifecycle. Unlike standard nodes, these enable determinstic
   startup, safe parameter reconfiguration, and remote pausing/restarting, safe,
   ordered initialisation, runtime reconfiguration, and pausing, providing 
   better control over system startup and error recovery compared to regular
   nodes.

   KEY ASPECTS OF LIFECYCLE NODES:
   - STATES:
      * UNCONFIGURED: Initial state; node is loaded but resources are not
        initialised.
      * INACTIVE: Configured but not active; publishers/timers are disabled.
      * ACTIVE: Fully functional; node is processing data and publishing.
      * FINALISED: Terminal state before destruction.
   - TRANSITIONS: Managed via `on_configure()`, `on_activate()`, 
     `on_deactivate()`, `on_cleanup()`, and `on_shutdown()`.
   - BENEFITS: Enables robust, deterministic system startup and allows nodes to
     be restarted or reconfigured on-the-fly.
   - USE CASES: Essential for complex systems like Nav2 (e.g., `map_server`,
     `controller_server`) to ensure hardware is properly initialised.
   - MANAGEMENT: Controlled via ROS2 CLI or external managers.       



-- Tree structure in URDF (unified Robot Description Format) define a robot as 
   a hierarchical model of links (rigid bodies) and joints (connections), where
   each link has one parent and zero or more child links. This structure 
   restricts robots to open-chain, branches geometries--no closed loops are 
   allowed. 

   - ROOT LINK: The base of the tree from which all other links branch.
   - LINKS: Physical components (e.g., chassis, wheels, arms) forming the nodes
     of the tree.
   - JOINTS: Connections defining the kinematics (parent-child relationship,
     type, and limits) between links.
   - CONSTRAINT: A link cannot have more than one parent, forming a tree rather
     than a graph.
   - LOOPS: URDF does not support closed kinematic loops, these must be modeled
     as a tree by breaking the loop.      

   Key joint types defining the structure include REVOLUTE (hinge), PRISMATIC
   (sliding), and FIXED (rigidly attached).







-- A closed kinematic chain in Gazebo is a mechanism where rigid bodies form a
   closed loop (e.g., parallel robots, grippers), creating constraints that
   require special modelling beyond standard tree-structure URDFs. Gazebo 
   supports these via SDF, often using `<mimic>` joints, plugins (like gz-sim),
   or by breaking the loop into a tree and closing it with specific constraints.

   KEY ASPECTS OF CLOSED CHAINS IN GAZEBO:
   - MODELLING CONSTRAINTS: Unlike open chains (tree structures), closed chains 
     cannot be easily represented in URDF because a link cannot have two parents
     . SDF (Simulation Description Format) is required for full graph support.
   - COMMON USE CASES: They are essential for modelling parallel robots, 
     2-finger grippers, and 4-bar linkages.   
   - SIMULATION IMPLEMENTATION:
      - GAZEBO CLASSIC: Uses plugins like `libgazebo_ros2_control.so` or `mimic`
        joints to enforce constraints.
      - GAZEBO SIM (Ignition): Does not directly support closed loop in SDF;
        instead, you must model the robot as a tree and use plugins to enforce
        closure, or use gz-physics to handle them.
   - SIMULATION CHALLENGES: They can cause numerical instability (numerical
     drift) because this simulation must satisfy strict geometric constraints at
     every time step.

   COMMON WORKAROUNDS:
   - MIMIC JOINTS: Setting one joint to mimic another to simulate dependency.
   - PLUGIN-BASED CONSTRAINTS: Utilising gazebo plugins to define the 
     closed-loop constraints explicitly.
   - CONVERTING TO TREE: Structuring the URDF to represent the loop as a 
     branched tree and handling the final connection via a plugin.  



-- In robotics, IMPEDANCE refers to a control strategy that regulates the 
   relationship between the force applied by a robot's manipulator 
   (end-effector) and its resulting motion (position/velocity) when
   interacting with the environment. Rather than forcing a robot to stick to a 
   strict path, impedance control makes it behave like a programmable 
   mass-spring damper system, allowing it to yield or resist in a controlled,
   safe, and natural way.



-- Joint-space control is a ROBOTIC CONTROL METHOD that directly specifies and
   regulates the position or motion of each individual robot joint (angles for
   revolute, extension for prismatic) to achieve a desired configuration, rather
   than focusing on the Cartesian posiion of the end-effector. It is often used
   for fast, point-to-point motion, as it avoids the need for complex, 
   computationally heavy INVERSE KINEMATICS during real-time movement.    



-- F/T (Force/Torque) sensor integration is the incorporation of sensors, 
   typically at the robot's wrist, to measure 6-axis forces ($F_x, F_y, F_z$)
   and torques ($M_x, M_y, M_z$) in real time. This allows robots to "feel,"
   enabling precise, adaptive control for tasks like assembly, polishing, and
   insertion.   



-- Admittance control is a robotic control strategy that translates external
   forces (measured by a sensor) into, usually, motion, by using a virtual
   mass-spring-damper model. It enables robots to behave compliantly, making it
   ideal for safe human-robot interaction, collaborative tasks, and teaching, 
   where the robot acts, or "admits," to external inputs.    



-- Protocol Buffers (Protobuf) is a language-neutral, platform-neutral, and
   extensible mechanism developed by Google for serialising structured data,
   frequently used in robotics for high-performance, compact communication
   between components. It enables efficient, binary-formatted data exchange,
   making it faster and smaller than JSON or XML, which is ideal fopr real-time
   robotic systems.    



-- Serialisation is the process of converting complex data structures or object
   states in memory into a protable format (like bytes, JSON, or XML) that can
   be easily stored in files/databases or transmitted across networks. This
   allows the object to be reconstructed later through deserialisation, 
   preserving the original data state.

---


---    
3. WHAT DOES `write_root_*` DO?
    You are correct--it writes to the SIMULATION BUFFER.

    In high-peformance simulation, you don't move objects by talking to the "UI"
    or the Scene Tree. That is too slow. Instead:

    1. `write_root_pose_to_sim`: This teleports the obejcts to the new XYZ
       coordinates and rotations you just calculated.    
    2. `write_root_velocity_to_sim`: This sets their speed. In this script, it
       usually      

                    https://gemini.google.com/app/d4324975ae77fb67

hi. thanks. can you now on the other hand explain to me how does run_articulation.py work... and how it differs from run_rigid_object.py? thanks!

                    https://gemini.google.com/app/d4324975ae77fb67

finally! please similarly do such too for run_deformable_object.py

                    https://gemini.google.com/app/d4324975ae77fb67